In [1]:
!pip install pandas numpy matplotlib pyarrow scikit-learn tkan "jax[cuda12]" keras_sig tkan sig_rnn --upgrade

# Exploring Signature-Based Gating in LSTM and GRU Networks

This notebook investigates the integration of path signatures into classical recurrent neural network architectures, focusing on their application in gating mechanisms. We explore two different approaches:

1. **Signature-Based LSTM**: A modified LSTM where the forget gate is controlled by path signatures instead of traditional linear transformations, while maintaining the standard computation for input and output gates.

2. **Signature-Based GRU**: An adaptation of the Gated Recurrent Unit where the reset gate computation is enhanced with path signatures, preserving the original update gate mechanics.

## Motivation

Traditional LSTM and GRU architectures use linear transformations followed by nonlinear activations for their gating mechanisms. However, path signatures provide a rich, hierarchical representation of sequential data that could potentially offer more sophisticated control over information flow in these networks. By incorporating signatures into specific gating mechanisms, we aim to:

- Capture higher-order temporal patterns in the input sequences
- Provide better control over long-term dependencies through signature-based forget/reset mechanisms
- Maintain the proven effectiveness of standard gates while enhancing critical gates with signature computations

## Implementation Details

The notebook implements these architectures using Keras 3's backend-agnostic framework, ensuring compatibility across TensorFlow, JAX, and PyTorch backends. Each implementation:
- Uses streaming signatures to maintain temporal coherence
- Normalizes signature values by sequence length
- Maintains the original architecture's core functionality while enhancing specific gates
- Computes signatures at the RNN level to optimize computational efficiency

## Usage

The implementations can be used as drop-in replacements for standard LSTM and GRU layers in any Keras model. They accept the same input formats and provide the same output structures as their traditional counterparts.

---

*Note: This research explores the potential benefits of combining classical RNN architectures with modern signature methods for enhanced temporal data processing.*

In [2]:
import os
BACKEND = 'jax' # You can use any backend here 
os.environ['KERAS_BACKEND'] = BACKEND

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Input, Flatten, GRU

from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error

from tkan import TKAN
from keras_sig import SigLayer

from sig_rnn import SignatureGRU,SignatureLSTM

import time

keras.utils.set_random_seed(1) 

N_MAX_EPOCHS = 1000
BATCH_SIZE = 128
early_stopping_callback = lambda : keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0.00001,
    patience=10,
    mode="min",
    restore_best_weights=True,
    start_from_epoch=6,
)
lr_callback = lambda : keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.25,
    patience=5,
    mode="min",
    min_delta=0.00001,
    min_lr=0.000025,
    verbose=0,
)
callbacks = lambda : [early_stopping_callback(), lr_callback(), keras.callbacks.TerminateOnNaN()]


# Data

In [3]:
df = pd.read_parquet('data.parquet')
df = df[(df.index >= pd.Timestamp('2020-01-01')) & (df.index < pd.Timestamp('2023-01-01'))]
assets = ['BTC', 'ETH', 'ADA', 'XMR', 'EOS', 'MATIC', 'TRX', 'FTM', 'BNB', 'XLM', 'ENJ', 'CHZ', 'BUSD', 'ATOM', 'LINK', 'ETC', 'XRP', 'BCH', 'LTC']
df = df[[c for c in df.columns if 'close' in c and any(asset in c for asset in assets)]]
df.columns = [c.replace(' close', '') for c in df.columns]
df = np.sqrt((df / df.shift() - 1.).dropna() ** 2)
display(df)

,BTC,ADA,XMR,EOS,CHZ,MATIC,TRX,ENJ,FTM,BNB,XLM,BUSD,ATOM,LTC,LINK,ETC,ETH,XRP,BCH
group,,,,,,,,,,,,,,,,,,,
2020-01-01 01:00:00,0.005469,0.006406,0.006955,0.010780,0.013347,0.003613,0.010574,0.000499,0.000000,0.007402,0.006462,0.000399,0.017070,0.008236,0.010942,0.009307,0.013735,0.006390,0.008609
2020-01-01 02:00:00,0.003683,0.005456,0.003119,0.002370,0.002242,0.007199,0.002242,0.007984,0.005525,0.003500,0.001328,0.000000,0.009324,0.005526,0.009702,0.006376,0.001607,0.002426,0.005771
2020-01-01 03:00:00,0.002463,0.004221,0.000444,0.005263,0.000979,0.007863,0.008949,0.012575,0.000916,0.002224,0.001327,0.000100,0.003464,0.007646,0.003999,0.008089,0.004968,0.001081,0.008053
2020-01-01 04:00:00,0.001071,0.001211,0.003774,0.003067,0.000419,0.002882,0.000752,0.000869,0.006416,0.001795,0.001328,0.000399,0.002317,0.001445,0.002008,0.001282,0.000000,0.001753,0.002382
2020-01-01 05:00:00,0.000962,0.003334,0.000000,0.000459,0.000838,0.001437,0.000000,0.009198,0.000923,0.000616,0.000665,0.000000,0.004413,0.004568,0.003506,0.000530,0.000768,0.003409,0.001649
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 19:00:00,0.000461,0.005663,0.004073,0.003472,0.002985,0.004732,0.001101,0.004141,0.002003,0.001217,0.000000,0.000000,0.000214,0.003146,0.003953,0.000633,0.002543,0.003514,0.000000
2022-12-31 20:00:00,0.000093,0.002011,0.000000,0.002307,0.000992,0.000916,0.000183,0.001237,0.000500,0.000000,0.004219,0.000100,0.001391,0.003421,0.000179,0.003802,0.000258,0.000584,0.004115
2022-12-31 21:00:00,0.001226,0.004817,0.002028,0.003452,0.003972,0.004190,0.000733,0.004942,0.003996,0.000405,0.001401,0.000100,0.002884,0.002273,0.003759,0.005682,0.001863,0.002625,0.004098


In [4]:
def generate_data(df, sequence_length, n_ahead):
    #Case without known inputs

    tmp_df = df.copy() #/ scaler_df
    def prepare_sequences(df, n_history, n_future):
        X, y = [], []
        num_features = df.shape[1]
        
        # Iterate through the DataFrame to create sequences
        for i in range(n_history, len(df) - n_future + 1):
            # Extract the sequence of past observations
            X.append(df.iloc[i - n_history:i].values)
            # Extract the future values of the first column
            y.append(df.iloc[i:i + n_future,0].values)
           
        return np.array(X), np.array(y)
    
    # Prepare sequences
    X, y = prepare_sequences(tmp_df, sequence_length, n_ahead)
    
    # Split the dataset into training and testing sets
    train_test_separation = int(len(X) * 0.8)
    X_train, X_test = X[:train_test_separation], X[train_test_separation:]
    y_train, y_test = y[:train_test_separation], y[train_test_separation:]

    max_X_train = np.max(X_train, axis=0)
    X_train /= max_X_train
    X_test /= max_X_train

    max_y_train = np.max(y_train, axis=0)
    y_train /= max_y_train
    y_test /= max_y_train
    
    y_train = y_train.reshape(y_train.shape[0], -1) 
    y_test = y_test.reshape(y_test.shape[0], -1)
    return X_train, X_test, y_train, y_test

In [5]:
n_aheads = [1, 6]
models = [
    "SignatureLSTM-2-2",
    "SignatureGRU-2-2",
    "SignatureLSTM-3-2",
    "SignatureGRU-3-2",
    "SignatureLSTM-3-3",
    "SignatureGRU-3-3",
    "SignatureLSTM-2_10-2",
    "SignatureGRU-2_10-2",
    "SignatureLSTM-3-3-3",
    "SignatureGRU-3-3-3",
    "SignatureLSTM-4-4",
    "SignatureGRU-4-4",
    "SignatureLSTM-3-3-flatten",
    "SignatureGRU-3-3-flatten",
    "GRU",
    "LSTM",
    "GRU-3",
    "LSTM-3",
    "GRU-flatten",
    "LSTM-flatten",
 ]

results = {model: {n_ahead: [] for n_ahead in n_aheads} for model in models}
results_rmse = {model: {n_ahead: [] for n_ahead in n_aheads} for model in models}
time_results = {model: {n_ahead: [] for n_ahead in n_aheads} for model in models}
for n_ahead in n_aheads:
    sequence_length = max(45, 5 * n_ahead)
    for run in range(5):
        X_train, X_test, y_train, y_test = generate_data(df, sequence_length, n_ahead)
        
        for model_id in models:

            if model_id == 'LSTM':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'GRU':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-2-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, return_sequences=True),
                    SignatureLSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-2-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, return_sequences=True),
                    SignatureGRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-2_10-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_input_size=10, return_sequences=True),
                    SignatureLSTM(100, signature_input_size=10, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-2_10-2':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_input_size=10, return_sequences=True),
                    SignatureGRU(100, signature_input_size=10, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'LSTM-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'GRU-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'LSTM-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=True),
                    LSTM(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'GRU-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=True),
                    GRU(100, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-3-3':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-4-4':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=4, return_sequences=True),
                    SignatureLSTM(100, signature_depth=4, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-4-4':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=4, return_sequences=True),
                    SignatureGRU(100, signature_depth=4, return_sequences=False),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureLSTM-3-3-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    SignatureLSTM(100, signature_depth=3, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            elif model_id == 'SignatureGRU-3-3-flatten':
                model = Sequential([
                    Input(shape=X_train.shape[1:]),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    SignatureGRU(100, signature_depth=3, return_sequences=True),
                    Flatten(),
                    Dense(units=n_ahead, activation='linear')
                ], name = model_id)
            else:
                raise ValueError

            optimizer = keras.optimizers.Adam(0.001)
            model.compile(optimizer=optimizer, loss='mean_squared_error', jit_compile=True)
            if run==0:
                model.summary()
                
            # Fit the model
            start_time = time.time()
            history = model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=N_MAX_EPOCHS, validation_split=0.2, callbacks=callbacks(), shuffle=True, verbose = False)
            end_time = time.time()
            time_results[model_id][n_ahead].append(end_time - start_time)
            # Evaluate the model on the test set
            preds = model.predict(X_test, verbose=False)
            r2 = r2_score(y_true=y_test, y_pred=preds)
            print(model_id, end_time - start_time, r2)
            rmse = root_mean_squared_error(y_true=y_test, y_pred=preds)
            results[model_id][n_ahead].append(r2)
            results_rmse[model_id][n_ahead].append(rmse)
    
            del model
            del optimizer
                

print('R2 scores')
print('Means:')
df_mean_r2 = pd.DataFrame({model_id: {n_ahead: np.mean(results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results.keys()})
df_mean_r2.to_csv('volatility_mean_r2.csv')
display(df_mean_r2)
df_mean_rmse = pd.DataFrame({model_id: {n_ahead: np.mean(results_rmse[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results_rmse.keys()})
df_mean_rmse.to_csv('volatility_mean_rmse.csv')
display(df_mean_rmse)
print('Std:')
df_std_r2 = pd.DataFrame({model_id: {n_ahead: np.std(results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results.keys()})
df_std_r2.to_csv('volatility_std_r2.csv')
display(df_std_r2)
df_std_rmse = pd.DataFrame({model_id: {n_ahead: np.std(results_rmse[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in results_rmse.keys()})
df_std_rmse.to_csv('volatility_std_rmse.csv')
display(df_std_rmse)
print('Training Times')
df_mean_time = pd.DataFrame({model_id: {n_ahead: np.mean(time_results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in time_results.keys()})
df_mean_time.to_csv('volatility_mean_time.csv')
display(df_mean_time)
df_std_time = pd.DataFrame({model_id: {n_ahead: np.std(time_results[model_id][n_ahead]) for n_ahead in n_aheads} for model_id in time_results.keys()})
df_std_time.to_csv('volatility_std_time.csv')
display(df_std_time)

Model: "SignatureLSTM-2-2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signature_lstm (SignatureLSTM)  │ (None, 45, 100)        │        39,195 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ signature_lstm_1                │ (None, 100)            │        63,900 │
│ (SignatureLSTM)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,196 (403.11 KB)

 Trainable params: 103,196 (403.11 KB)

 Non-trainable params: 0 (0.00 B)

...

,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,0.009897,0.007154,0.004329,0.009734,0.010296,0.010704,0.005693,0.005507,0.002944,0.008688,0.003306,0.004172,0.008505,0.005417,0.003274,0.001750,0.007135,0.010616,0.013127,0.003572
6,0.003079,0.007738,0.008438,0.011692,0.006814,0.003816,0.005398,0.003905,0.005677,0.006670,0.004753,0.002668,0.006883,0.004493,0.007002,0.007469,0.015485,0.004716,0.002235,0.004891


,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,0.000153,0.000111,0.000067,0.000151,0.000159,0.000165,0.000089,0.000085,0.000046,0.000134,0.000051,0.000065,0.000131,0.000083,0.000051,0.000027,0.000111,0.000164,0.000202,0.000055
6,0.000047,0.000117,0.000128,0.000173,0.000103,0.000058,0.000082,0.000059,0.000087,0.000101,0.000072,0.000041,0.000105,0.000068,0.000107,0.000113,0.000232,0.000072,0.000034,0.000074


Training Times


,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,80.623121,39.003515,86.940609,43.147981,86.826948,42.196554,85.117001,42.385486,122.056018,55.356374,93.154561,48.743154,116.281870,42.767124,17.030538,14.881003,24.406852,21.060413,16.607709,14.420431
6,87.688067,39.659723,93.133675,42.786932,91.791450,42.109399,92.136857,39.378980,130.076298,57.531963,98.962551,46.758729,95.983314,40.368893,15.811198,14.266801,23.204025,20.577898,17.957346,14.821177


,SignatureLSTM-2-2,SignatureGRU-2-2,SignatureLSTM-3-2,SignatureGRU-3-2,SignatureLSTM-3-3,SignatureGRU-3-3,SignatureLSTM-2_10-2,SignatureGRU-2_10-2,SignatureLSTM-3-3-3,SignatureGRU-3-3-3,SignatureLSTM-4-4,SignatureGRU-4-4,SignatureLSTM-3-3-flatten,SignatureGRU-3-3-flatten,GRU,LSTM,GRU-3,LSTM-3,GRU-flatten,LSTM-flatten
1,2.354491,0.737338,3.517930,1.563352,2.890416,1.347770,4.341440,0.773150,3.502773,1.628675,5.388082,2.223184,2.602397,1.377568,1.404687,1.362235,3.274173,1.559217,1.148858,0.886984
6,2.572248,1.588180,3.503215,0.693861,5.465062,1.191648,3.292081,0.685502,2.736443,4.436068,4.741691,2.342841,5.572618,1.058047,0.362412,0.119576,1.488962,1.033417,2.602842,1.021011
